# 📘 EarthDaily Agriculture — `HistoryManager` Service Dev Notebook

Exercise the `HistoryManager` service against the live History API (`/accounts/history/v1/history-entries`). Runs GET (list / by-id / count) and POST (create / save_user_entry) end-to-end.

> ⚠️ **No DELETE endpoint.** Every POST appends a new row. This notebook uses a recognizable test type (`TEST_HISTORY_MANAGER_DEMO`) so any rows it creates are easy to spot in later sweeps.

> ⚠️ **Default env is `preprod`.** Flip `ENV` in Step 1 to `"prod"` only after you've verified the flow on preprod.

## ✅ Step 0: Bootstrap

In [ ]:
# Bootstrap: ensure src/ is on sys.path for earthdaily.agriculture imports
import sys
from pathlib import Path

_src = str(Path().resolve().parent / "src")
if _src not in sys.path:
    sys.path.insert(0, _src)

from earthdaily.agriculture.notebook_setup import init
init()

## ✅ Step 1: Init `WorkflowManager` + `HistoryManager`

`HistoryManager` reuses the token bootstrapped by `WorkflowManager` (so token auto-refresh keeps working via `workflow_ref`).

In [ ]:
from earthdaily.agriculture.services.workflow_manager import WorkflowManager
from earthdaily.agriculture.services.history_manager import HistoryManager

# Safe default — flip to "prod" only after verifying the flow.
ENV = "preprod"

manager = WorkflowManager(ENV, log_to_console=True, log_level="INFO")

hm = HistoryManager(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config={"env": ENV},
    workflow_ref=manager,
)
print(f"Base URL: {hm.base_url}")

## ✅ Step 1.5: Resolve `USER_ID` from a login *(optional)*

> **Skip this step for a smoke test.** The History API uses the bearer token's identity when no `userid` is supplied — `USER_ID = None` returns your own entries. Run this step only when you need to query another user's history (CS / support flow).

Look up the user id via `EntityManager.get_grower_id(LOGIN)` (which delegates to `UserManager.get_user_by_login` under the hood). The id returned here is exactly what the History API stores in the `userid` field — same format on the wire.

Set `LOGIN` to your own login (or a test user's login on preprod) before running. `get_grower_id` is named for the GROWER user type but the underlying lookup is generic — works for AGRONOMIST logins too.

In [ ]:
# SKIP this cell for a smoke test — just run the next cell with `USER_ID = None`.
# Otherwise: resolve USER_ID from a login below.

from earthdaily.agriculture.services.user_management import UserManager
from earthdaily.agriculture.services.entity_management import EntityManager

LOGIN = "YOUR_LOGIN_HERE"  # the login to resolve; only used if you want another user's history

um = UserManager(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config={"env": ENV},
    workflow_ref=manager,
)
em = EntityManager(
    bearer_token=manager.bearer_token,
    token_expiration=manager.token_expiration,
    config={"env": ENV},
    workflow_ref=manager,
    user_manager=um,
)

USER_ID = em.get_grower_id(LOGIN)
if not USER_ID:
    raise RuntimeError(
        f"Could not resolve user id for login {LOGIN!r} — verify the login is correct "
        f"and exists in env={ENV!r}."
    )
print(f"Resolved USER_ID = {USER_ID!r} for login {LOGIN!r}")

## 🔍 Step 2: GET — `list_entries`

If you skipped Step 1.5, set `USER_ID = None` in the cell below — the bearer's identity is used and you'll get your own entries. If you ran Step 1.5, `USER_ID` is already set from your `LOGIN`.

In [ ]:
# Smoke test: leave the line below uncommented and `USER_ID = None`.
# Querying another user's history: set USER_ID via Step 1.5 instead and skip this line.
USER_ID = locals().get("USER_ID", None)

rows = hm.list_entries(user_id=USER_ID, limit=20)
who = USER_ID or "<authenticated bearer>"
print(f"Found {len(rows)} entries for user {who!r}")
for r in rows:
    print(f"  - id={str(r.get('id'))[:24]:24s}  type={r.get('type')!r}")

### Filter by `type=CUSTOM_ANALYTIC_UNIQUEAPP`

This is the in-scope type that carries the user's custom analytic definitions. The inner shape is `{version, configurations: [...]}`.

In [ ]:
custom_rows = hm.list_entries(
    user_id=USER_ID,
    type="CUSTOM_ANALYTIC_UNIQUEAPP",
)
print(f"{len(custom_rows)} CUSTOM_ANALYTIC_UNIQUEAPP entries")

if custom_rows:
    first = custom_rows[0]
    print(f"\nid           : {first.get('id')}")
    print(f"element_id   : {first.get('element_id')}")
    print(f"data keys    : {list(first.get('data', {}).keys())}")

    configs = first.get("data", {}).get("configurations", [])
    print(f"# configs    : {len(configs)}")
    for c in configs[:5]:
        schema = c.get("analyticFabricConfiguration", {}).get("schemaId")
        print(f"  - {c.get('id')}: {c.get('name')!r}  schema={schema}")

## 📊 Step 3: `.to_dataframe()` — envelope only

`data` stays nested as an object column (not exploded). Useful for scanning many entries without digging into per-type payloads.

In [ ]:
df = rows.to_dataframe()
df

## 🔍 Step 4: GET — single entry by id

In [ ]:
if rows:
    target_id = rows[0]["id"]
    single = hm.get_entry(target_id)
    print(f"Fetched entry id={target_id}")
    print(f"  type        : {single.get('type')}")
    print(f"  element_id  : {single.get('element_id')}")
    print(f"  user_id     : {single.get('user_id')}")
    print(f"  data preview: {str(single.get('data'))[:160]}...")
else:
    print("No entries for this user — skip Step 4")

## 🔢 Step 5: HEAD — `count_entries`

Cheap counter via the HEAD endpoint; reads `X-Total-Count` from the response header.

> ⚠️ The OpenAPI doesn't name the count header explicitly. If counts come back as 0 here while `list_entries()` returns rows, the server is using a different header name — see `HistoryManager.count_entries`.

In [ ]:
total = hm.count_entries(user_id=USER_ID)
print(f"Total entries for user {USER_ID!r}: {total}")

total_custom = hm.count_entries(user_id=USER_ID, type="CUSTOM_ANALYTIC_UNIQUEAPP")
print(f"...of which CUSTOM_ANALYTIC_UNIQUEAPP: {total_custom}")

## ⚡ Step 6: POST — `create_entry`

Writes a **real row** to the History API. Uses a clearly-tagged `type=TEST_HISTORY_MANAGER_DEMO` so any rows this notebook leaves behind are easy to grep for later.

> ⚠️ The API has **no DELETE**. Every run of this cell appends another row.

In [ ]:
TEST_TYPE = "TEST_HISTORY_MANAGER_DEMO"

new_entry = hm.create_entry(
    type=TEST_TYPE,
    data={
        "version": 1,
        "source": "EDAgriculture_HistoryManager_Service_Function_Dev.ipynb",
        "configurations": [
            {
                "id": "demo-1",
                "name": "Demo from notebook",
            }
        ],
    },
    element_id=TEST_TYPE,
    user_id=USER_ID,
)
print("POST OK — server returned:")
print(new_entry)

### Verify the POST landed

In [ ]:
roundtrip = hm.list_entries(user_id=USER_ID, type=TEST_TYPE)
print(f"{len(roundtrip)} {TEST_TYPE} entries for user {USER_ID!r}")
for r in roundtrip:
    src = r.get("data", {}).get("source")
    ver = r.get("data", {}).get("version")
    print(f"  - id={r.get('id')!r}  version={ver!r}  source={src!r}")

## 🚨 Step 7: `save_user_entry` — warn-when-prior-exists

After Step 6 there's at least one `TEST_HISTORY_MANAGER_DEMO` row. `save_user_entry` detects this and logs a warning, then POSTs anyway (no DELETE means it can't actually replace). Useful sanity check that the warning fires when expected.

In [ ]:
appended = hm.save_user_entry(
    user_id=USER_ID,
    type=TEST_TYPE,
    data={"version": 2, "note": "second-pass append"},
    element_id=TEST_TYPE,
)
print("save_user_entry returned:")
print(appended)
print()
print("Re-list to confirm both rows present:")
for r in hm.list_entries(user_id=USER_ID, type=TEST_TYPE):
    print(f"  - id={r.get('id')!r}  version={r.get('data', {}).get('version')!r}")

## 🧹 Cleanup

There's no DELETE on the History API, so the `TEST_HISTORY_MANAGER_DEMO` rows this notebook created will stay in the user's history until EarthDaily clears them at the platform level. Two practical options:

1. **Filter them out of prod queries** with `filter_expr="Type!='TEST_HISTORY_MANAGER_DEMO'"` (or any client-side filter on the listed rows).
2. **Use a dedicated test user** in preprod so the prod user's history stays clean.